# MODELO DE CLASIFICACIÓN — Informalidad laboral (ENAHO 2023)

**Target:** `y_clasif` = 1 si el asalariado es informal, 0 si es formal (derivado de `ocupinf`, calculada por INEI). Balance: 59% informal / 41% formal.
**Universo:** 25 232 asalariados ocupados con ingreso declarado (ENAHO 2023).
**Predictores (solo sociodemográficos, sin variables de empleo/ingreso):** edad, sexo, parentesco, nivel educativo, lengua materna, dominio, área y campo de estudio. `ocupinf` es el target, no entra como feature.

**Cómo usarlo:**
1. Archivo → Subir copia en Drive (o abrir en Colab).
2. Entorno de ejecución → Ejecutar todo.
3. La base se lee de la carpeta compartida `CODIGOS_ENAPRES/BASE_ENAHO/2023` (ruta en la celda de Drive).
4. Al final se descarga el modelo entrenado (`clasificador_informalidad.joblib`) para el despliegue en Streamlit.

## Importing the libraries

In [ ]:
# Instala lo que no viene por defecto en Colab
!pip install -q pyreadstat joblib xgboost

# Cómo importar las librerías
import os
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadstat
import joblib

## 1. Lectura de datos

Montamos Google Drive y cargamos los 3 módulos `.sav` (Miembros del hogar, Educación, Empleo e Ingresos) desde la carpeta compartida.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Ruta de la carpeta compartida (cámbiala si tu Drive la tiene en otro lugar)
RUTA_BASE = '/content/drive/My Drive/CODIGOS_ENAPRES/BASE_ENAHO/2023'
print('Ruta base:', RUTA_BASE, '| Existe:', os.path.isdir(RUTA_BASE))

In [ ]:
def norm(c):
    return unicodedata.normalize('NFKD', str(c)).encode('ascii', 'ignore').decode('ascii').lower()

ARCHIVOS = {
    'mod02': RUTA_BASE + '/mod02/906-Modulo02/Enaho01-2023-200.sav',
    'mod03': RUTA_BASE + '/mod03/906-Modulo03/Enaho01A-2023-300.sav',
    'mod05': RUTA_BASE + '/mod05/906-Modulo05/Enaho01a-2023-500.sav',
}

modulos = {}
for nombre, ruta in ARCHIVOS.items():
    df, _ = pyreadstat.read_sav(ruta)
    df.columns = [norm(c) for c in df.columns]
    modulos[nombre] = df
    print(nombre, df.shape)

### Merge y filtros (spec: 86 654 → 59 447 → 25 244 → 25 232)

1. Inner join por llave persona `(mes, conglome, vivienda, hogar, codperso)`
2. PEA ocupada (`ocupinf` no nulo)
3. Asalariados con ingreso (`i524a1 > 0`)
4. Sin nulos en predictores

In [ ]:
LLAVES = ['mes', 'conglome', 'vivienda', 'hogar', 'codperso']

sub02 = modulos['mod02'][LLAVES + ['p203', 'p207', 'p208a', 'estrato', 'dominio']]
sub03 = modulos['mod03'][LLAVES + ['p301a', 'p300a', 'p301a1']]
sub05 = modulos['mod05'][LLAVES + ['ocupinf', 'i524a1']]

paso1 = sub05.merge(sub02, on=LLAVES, how='inner', validate='one_to_one')
paso1 = paso1.merge(sub03, on=LLAVES, how='inner', validate='one_to_one')
paso2 = paso1[paso1['ocupinf'].notna()].copy()
paso3 = paso2[paso2['i524a1'] > 0].copy()
print('1. Join completo:', len(paso1))
print('2. PEA ocupada :', len(paso2))
print('3. Con ingreso :', len(paso3))

## 2. Transformaciones de datos

- `area`: urbano (1) si estrato ≤ 5, rural (0) si ≥ 6
- `parentesco`: Jefe / Conyuge / Hijo / Otro_familiar
- `lengua_materna`: Castellano / Quechua / Aimara / Otra_nativa / Extranjera_otra
- `campo_estudio`: 8 campos de la carrera (de `p301a1`) + Sin_carrera
- `y_clasif`: 1 = informal, 0 = formal

In [ ]:
df = paso3[['p208a', 'p207', 'p203', 'p301a', 'p300a', 'p301a1',
            'dominio', 'estrato', 'ocupinf', 'i524a1']].copy()

df['area'] = (df['estrato'] <= 5).astype(int)
df['parentesco'] = df['p203'].map({1: 'Jefe', 2: 'Conyuge', 3: 'Hijo'}).fillna('Otro_familiar')

nativas = {3, 10, 11, 12, 13, 14, 15}
df['lengua_materna'] = np.where(
    df['p300a'].isin({1, 2, 4}),
    df['p300a'].map({1: 'Quechua', 2: 'Aimara', 4: 'Castellano'}),
    np.where(df['p300a'].isin(nativas), 'Otra_nativa', 'Extranjera_otra'))

MAPA_CAMPO = {1: 'Educacion', 2: 'Ciencias', 3: 'Admin_Contab_Derecho',
              4: 'Computacion_Informatica', 5: 'Ingenieria_Tecnicas',
              6: 'Agropecuaria', 7: 'Salud'}
d = pd.to_numeric(df['p301a1'], errors='coerce')
dig = (d // 100000).astype('Int64')
campo = dig.map(MAPA_CAMPO).fillna('Artes_Otras')
df['campo_estudio'] = campo.where(d.notna(), 'Sin_carrera')

df = df.rename(columns={'p208a': 'edad', 'p207': 'sexo', 'p301a': 'nivel_educ'})
df['y_clasif'] = (df['ocupinf'] == 1).astype(int)

# Eliminamos nulos ANTES de convertir a texto (evita errores con valores faltantes)
df = df.dropna(subset=['edad', 'sexo', 'parentesco', 'nivel_educ', 'lengua_materna',
                       'dominio', 'area']).copy()
print('Dataset final (4. Sin nulos en predictores):', df.shape)

# Convertimos las categoricas a texto para que el OneHotEncoder use las mismas
# categorias en la app de Streamlit (evita errores de tipo int/float)
df['sexo'] = df['sexo'].astype(int).astype(str)
df['nivel_educ'] = df['nivel_educ'].astype(int).astype(str)
df['dominio'] = df['dominio'].astype(int).astype(str)
df['area'] = df['area'].astype(str)
df = df.dropna(subset=['edad', 'sexo', 'parentesco', 'nivel_educ', 'lengua_materna',
                       'dominio', 'area']).copy()
print('Dataset final (4. Sin nulos en predictores):', df.shape)
print('Balance target:')
print(df['y_clasif'].value_counts().to_string())
print('Porcentaje informal:', round(df['y_clasif'].mean() * 100, 1), '%')


## Exploración descriptiva

Distribución de las variables y su relación con la informalidad.

In [ ]:
print(df[['edad']].describe().round(2))
print()
print('Porcentaje por categoría:')
for col in ['sexo', 'parentesco', 'nivel_educ', 'lengua_materna', 'dominio', 'area', 'campo_estudio']:
    print('---', col)
    print(df[col].value_counts(normalize=True).round(3).to_string())
    print()
print('Nulos totales:', int(df.isna().sum().sum()))

In [ ]:
# Relación de cada categórica con el target (proporción de informales por categoría)
for col in ['sexo', 'parentesco', 'area', 'campo_estudio']:
    tabla = df.groupby(col)['y_clasif'].mean().round(3)
    print('---', col, '(proporción informal)')
    print(tabla.to_string())
    print()

## 3. Pre Procesamiento de datos

Separamos las variables numéricas y categóricas y creamos el `ColumnTransformer`. **OJO: ninguna variable de empleo/ingreso entra como predictor; `ocupinf` es el target.**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Variables de X (solo sociodemográficas + campo de estudio)
numericas = ['edad']
categoricas = ['sexo', 'parentesco', 'nivel_educ', 'lengua_materna', 'dominio', 'area', 'campo_estudio']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas)
])

## 4. Data X e Y / Train y Test

Split estratificado por el target (para mantener el 59/41 en ambos conjuntos).

In [ ]:
X = df[numericas + categoricas]
y = df['y_clasif']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=123, stratify=y)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Proporción informal en train:', round(y_train.mean(), 3), '| en test:', round(y_test.mean(), 3))

## 5. CV y Pipeline

Validación cruzada estratificada repetida (5 folds × 2 repeticiones) y dos pipelines: Random Forest (con `class_weight='balanced'`) y XGBoost.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=123)

pipeline_rf = Pipeline([
    ('preproc', preprocessor),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=123))
])

pipeline_xg = Pipeline([
    ('preproc', preprocessor),
    ('clf', XGBClassifier(random_state=123, eval_metric='logloss'))
])

## 6. Tunning (GridSearchCV)

In [ ]:
rf_grid = {
    'clf__n_estimators': [100, 300],
    'clf__max_depth': [10, 20],
    'clf__min_samples_leaf': [2, 5]
}

xg_grid = {
    'clf__max_depth': [3, 5],
    'clf__n_estimators': [100, 300],
    'clf__learning_rate': [0.05, 0.1]
}

In [ ]:
rf_tunned = GridSearchCV(pipeline_rf, rf_grid, cv=cv, n_jobs=-1)
rf_tunned.fit(X_train, y_train)
print('MEJORES PARÁMETROS RANDOM FOREST:')
print(rf_tunned.best_params_)

In [ ]:
xg_tunned = GridSearchCV(pipeline_xg, xg_grid, cv=cv, n_jobs=-1)
xg_tunned.fit(X_train, y_train)
print('MEJORES PARÁMETROS XGBOOST:')
print(xg_tunned.best_params_)

## 7. Métricas (train vs test)

Accuracy y ROC-AUC en train y test, para cada modelo. Atención a la clase FORMAL (minoritaria, 41%).

In [ ]:
resultados = {}
for nombre, modelo in [('Random Forest', rf_tunned), ('XGBoost', xg_tunned)]:
    resultados[nombre] = {
        'Acc train': metrics.accuracy_score(y_train, modelo.predict(X_train)),
        'Acc test': metrics.accuracy_score(y_test, modelo.predict(X_test)),
        'AUC train': metrics.roc_auc_score(y_train, modelo.predict_proba(X_train)[:, 1]),
        'AUC test': metrics.roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1]),
    }
pd.DataFrame(resultados).round(4)

In [ ]:
# Matriz de confusión en TEST del mejor por AUC
mejor = xg_tunned if resultados['XGBoost']['AUC test'] >= resultados['Random Forest']['AUC test'] else rf_tunned
print('MODELO GANADOR:', 'XGBoost' if mejor is xg_tunned else 'Random Forest')
print()
probs = mejor.predict(X_test)
confusion_matrix2 = pd.crosstab(y_test, probs)  # reales, predichos
confusion_matrix2.columns = ['Pred Formal (0)', 'Pred Informal (1)']
confusion_matrix2.index = ['Real Formal (0)', 'Real Informal (1)']
confusion_matrix2

In [ ]:
# Reporte por clase del modelo ganador (test)
print(classification_report(y_test, mejor.predict(X_test),
                           target_names=['Formal (0)', 'Informal (1)']))

## 8. Importancia de variables

Importancia por permutación del modelo ganador.

In [ ]:
from sklearn.inspection import permutation_importance

importancia = permutation_importance(estimator=mejor, X=X_train, y=y_train,
                                     n_repeats=5, scoring='accuracy', random_state=123)
df_importancia = pd.DataFrame({'importances_mean': importancia['importances_mean'],
                               'importances_std': importancia['importances_std']})
df_importancia['feature'] = X_train.columns
df_importancia = df_importancia.sort_values('importances_mean', ascending=True)

fig, ax = plt.subplots(figsize=(4, 5))
ax.barh(df_importancia['feature'], df_importancia['importances_mean'],
        xerr=df_importancia['importances_std'], align='center', alpha=0)
ax.plot(df_importancia['importances_mean'], df_importancia['feature'],
        marker='D', linestyle='', alpha=0.8, color='r')
ax.set_title('Importancia de los predictores (train)')
ax.set_xlabel('Incremento del error tras la permutación')
plt.tight_layout()
plt.show()

## 9. Despliegue

Guardamos el pipeline ganador (con todas las transformaciones incluidas), lo cargamos y probamos con una observación nueva: predicción y probabilidades por clase.

In [ ]:
# Guardar el modelo
joblib.dump(mejor, 'clasificador_informalidad.joblib')

# Cargar el modelo
clasificador = joblib.load('clasificador_informalidad.joblib')
print('Modelo guardado y cargado correctamente.')

In [ ]:
# Nueva observación (una persona asalariada de ejemplo)
obs = pd.DataFrame([{
    'edad': 30, 'sexo': '2', 'parentesco': 'Hijo', 'nivel_educ': '6',
    'lengua_materna': 'Castellano', 'dominio': '8', 'area': '1',
    'campo_estudio': 'Sin_carrera'
}])
obs


In [ ]:
# Predicción y probabilidades por clase
probs2 = clasificador.predict_proba(obs)
probs2_df = pd.DataFrame(probs2, columns=['Prob_Formal', 'Prob_Informal'])
probs2_df['Predicción'] = clasificador.predict(obs)
probs2_df['Predicción'] = probs2_df['Predicción'].replace({0: 'Formal', 1: 'Informal'})
probs2_df

### Descargar el modelo para el despliegue en Streamlit

In [ ]:
from google.colab import files
files.download('clasificador_informalidad.joblib')
print('Fin del notebook.')